<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L02-evaluating-across-languages/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L02-evaluating-across-languages/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L02-evaluating-across-languages/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L02-evaluating-across-languages/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L02 · Evaluating across languages: the token tax meets the benchmark

**You will build:** the measuring instruments for a multilingual evaluation — token fertility,
cost per correct answer, item-level bootstrap intervals, a paired comparison that refuses
mismatched items, macro and micro aggregation, a chance-level flag, a translation-artefact
detector and an items-per-language power curve — and use them on a simulated system whose
true accuracy in every language is planted, so every effect is measured against a known answer.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download, no API key
· **Prerequisites:** `T00-L01-the-8gb-track`, `T03-L01-bpe-from-scratch`

By the end you will be able to:

1. Implement token fertility per word and per character, and measure the token tax on six
   generated scripts, including what a per-byte figure hides.
2. Implement cost per correct answer and the context-budget effect, and measure what one
   correct answer costs in each language.
3. Implement item-level bootstrap intervals (per language, for a macro-average, and paired
   between two systems) that refuse mismatched items.
4. Measure how macro versus micro averaging reorders two systems, and flag a language whose
   interval reaches chance that an aggregate would hide.
5. Quantify a planted translation artefact with a matched native/translated comparison, and
   measure how many items per language stop adjacent languages' ranking from flipping.

No language model is involved. Every "system" below is a seeded function with a known accuracy,
which is what lets the lesson tell a real effect from a mistake in the measurement.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `T03-L01`

**This lesson assumes you can already:**

- tell a str from a bytes object in Python, and compute len(text) against len(text.encode('utf-8'))
- compute an accuracy as the mean of 0/1 outcomes, and say why a smaller sample makes it less certain
- draw random indices from numpy's default_rng and use them to index an array
- read a table of per-group results and say which differences you would not trust without an interval

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Unicode HOWTO](https://docs.python.org/3/howto/unicode.html) · Python documentation · *free* — characters, code points and UTF-8 bytes in Python, the reason a per-byte figure hides the token tax that exercise 1 measures per word and per character
- [Language Model Tokenizers Introduce Unfairness Between Languages](https://arxiv.org/abs/2305.15425) · Petrov, La Malfa, Torr and Bibi (arXiv) · *free* — the paper behind the token-tax and cost-per-language measurements: the same text can need many times more tokens in another language, with cost and context-window consequences
- [Translation Artifacts in Cross-lingual Transfer Learning](https://arxiv.org/abs/2004.04721) · Artetxe, Labaka and Agirre (arXiv; EMNLP 2020) · *free* — how translating a training or test set introduces subtle artifacts that change model scores, so translated and native test sets are not interchangeable: the effect exercise 9 plants and detects
- [With Little Power Comes Great Responsibility](https://arxiv.org/abs/2010.06595) · Card, Henderson, Khandelwal, Jia, Mahowald and Jurafsky (arXiv; EMNLP 2020) · *free* — why NLP comparisons are often underpowered, and how many items it takes to detect a real difference, which is the question the items-per-language power curve answers
- [An Introduction to the Bootstrap](https://www.routledge.com/An-Introduction-to-the-Bootstrap/Efron-Tibshirani/p/book/9780412042317) · Bradley Efron and R. J. Tibshirani (Chapman & Hall/CRC) · *paid* — the standard text on the bootstrap; chapter 6 (the bootstrap estimate of standard error) and chapter 13 (percentile confidence intervals) cover what exercises 3 to 5 compute

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import signal
import statistics
import sys
import threading
import time
import traceback
from typing import Callable, NamedTuple

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

SEED = 20260929          # fixed before any output was looked at; nothing below searches seeds

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}  # label -> "passed" | "failed" | "not started"

# The exercises, in the order you meet them. The progress board at the foot of the notebook is
# built from this, and a demo that is waiting on one names it from here.
_EXERCISES = {
    "exercise 1": "fertility()",
    "exercise 2": "cost_per_correct() and units_in_window()",
    "exercise 3": "bootstrap_ci()",
    "exercise 4": "macro_ci()",
    "exercise 5": "align_paired()",
    "exercise 6": "paired_diff_ci()",
    "exercise 7": "aggregate()",
    "exercise 8": "weak_languages()",
    "exercise 9": "origin_gap()",
    "exercise 10": "rank_flip_rate()",
}


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check runs under a time limit far above what a right answer needs (the whole lesson runs in a
# few seconds); a demo gets a longer one.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 10

def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others.

    A demo names the exercises it `needs`. Until each of them has passed its check, the demo
    says which one it is waiting for and skips, rather than failing half-way through its
    output. Every outcome is recorded in `_STATUS`, which the progress board at the foot reads.
    A check still running after `_CHECK_SECONDS` (a demo: `_DEMO_SECONDS`) is stopped and
    counts as failed, so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        todo = ", ".join(f"{name} ({_EXERCISES[name]})" for name in waiting)
        print(f"{label}: skipped — this needs {todo} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        print(f"{label}: not implemented yet — fill in {stub}() above, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that was already running (the grader's own
    time limit) keeps its deadline if it is due first and is put back, with the time it had
    left, afterwards. A correct answer finishes far inside the limit, and nothing that is
    printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop that keeps "
               "drawing until something happens may never end")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, marked passed, failed or not started, then the tally."""
    width = max(len(what) for what in _EXERCISES.values())
    print("progress board")
    for label, what in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<11} {what:<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label, state in _STATUS.items() if state == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went wrong "
              "in its own cell above, and every exercise heading has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise heading has hints you can open.")

## 1. What is simulated, and what is real

A real multilingual evaluation has two layers. The **system** is a language model, and here
it is a stand-in: for each language and each item it answers correctly with a probability
that this lesson *planted*, so the truth is known. The **instruments** are real: the
tokenizer, the cost arithmetic, the bootstrap, the aggregation and the paired comparison are
exactly the code you would point at a real model's answers. A wrong instrument shows up here
as a gap between what you measured and what was planted.

The languages are **generated**, not real. Each of six "languages" has its own script (two
use Latin letters) and a lexicon of syllable-built words, and item *k* in every language means
the same thing, so the text is parallel. Their names are labels for slots in the simulation:
`latn-a` is a large Latin-script language, `cyrl` Cyrillic, `hani` Han characters, `deva`
Devanagari, `arab` Arabic and `latn-b` a small second Latin-script language.

The generated Han text separates its words with spaces, like the other five, so that one
word-splitting rule applies to all of them: a simplification of the simulation.

Nothing here says how any real model treats any real language. The phenomena it reproduces
— tokenizer disparity, translated-benchmark artefacts, averaging and sample-size traps — are
documented in the papers listed in `claims.yaml`, each with a verbatim quote and the date
it was read (29 September 2026).

In [ ]:
# The world of the simulation. Everything downstream is derived from these constants.
LANGS = ("latn-a", "cyrl", "hani", "deva", "arab", "latn-b")   # largest to smallest benchmark
LABEL = {"latn-a": "Latin (large)", "cyrl": "Cyrillic", "hani": "Han", "deva": "Devanagari",
         "arab": "Arabic", "latn-b": "Latin (small)"}

SYLLABLES = {
    "latn-a": [c + v for c in "bdfgklmnprstvz" for v in "aeiou"],
    "latn-b": [c + v for c in "chjkmnpswy" for v in "aiu"] + [c + v + "n" for c in "mstk" for v in "aiu"],
    "cyrl":   [c + v for c in "бвгдзклмнпрстх" for v in "аеиоуы"],
    "deva":   [c + v for c in "कगचजतदनपबमयरलवस" for v in ["", "ा", "ि", "ी", "ु", "े", "ो"]],
    "arab":   [c + v for c in "بتثجحخدرزسشصطعغفقكلمنهوي" for v in ["", "ا"]],
    "hani":   list("人大小山水火木金土日月天地上下中国门口手心目耳石田力刀工马牛羊鸟鱼云雨风花草书学生年"),
}
LEX_SIZE = 300           # distinct words per language; word k means the same in every language
TRAIN_WORDS = 20000      # words in the tokenizer's training text
VOCAB_SIZE = 700         # merge budget; training stops earlier if no pair occurs twice
TRAIN_SHARE = {"latn-a": 0.55, "latn-b": 0.15, "cyrl": 0.12, "deva": 0.08, "arab": 0.06, "hani": 0.04}
PASSAGE_WORDS = 400      # words in each language's held-out parallel passage

_WHITESPACE = frozenset(b" \t\n\r\f\v")
_ZIPF = 1.0 / np.arange(1, LEX_SIZE + 1)
_ZIPF = _ZIPF / _ZIPF.sum()


def make_lexicon(lang: str, n_words: int, seed: int) -> list[str]:
    """`n_words` distinct words, each one to three syllables (one or two characters for Han)."""
    rng = np.random.default_rng(seed)
    syllables = SYLLABLES[lang]
    longest = 2 if lang == "hani" else 3
    words, seen = [], set()
    while len(words) < n_words:
        k = int(rng.integers(1, longest + 1))
        word = "".join(syllables[int(i)] for i in rng.integers(0, len(syllables), size=k))
        if word not in seen:
            seen.add(word)
            words.append(word)
    return words


LEXICONS = {lang: make_lexicon(lang, LEX_SIZE, SEED + i) for i, lang in enumerate(LANGS)}


def sample_concepts(n: int, seed: int) -> np.ndarray:
    """`n` word indices drawn with Zipf weights: a few words are frequent, most are rare."""
    return np.random.default_rng(seed).choice(LEX_SIZE, size=n, p=_ZIPF)


def render(lang: str, concepts) -> str:
    """The same concepts written in `lang`: parallel text, one space between words."""
    return " ".join(LEXICONS[lang][int(c)] for c in concepts)


if __name__ == "__main__":
    _demo = sample_concepts(5, SEED)
    print("the same five concepts, written in each generated language:")
    for _lang in LANGS:
        _text = render(_lang, _demo)
        print(f"  {_lang:7} {_text}   ({len(_text)} characters, {len(_text.encode('utf-8'))} UTF-8 bytes)")

## 2. The tokenizer (given)

This is `T03-L01`'s byte-level BPE, carried over so this lesson runs on its own. The trainer
is the same algorithm with the same tie-break, rewritten to count each *distinct* word once and
multiply by its frequency: merges never cross a word, so it produces the same merge list on
the same words, only faster. The encoder replays the merges in training order.

One deliberate difference from T03: `count_tokens` does **not** charge for the spaces between
words. A space would add one more token per word in every language alike, which would only
blur the comparison, so it is left out, and the character counts below leave the spaces out too.

The tokenizer is trained on a mixture that mirrors the benchmark's own imbalance: most of the
words are `latn-a`, and `hani` gets the fewest. Run the cell to train it.

In [ ]:
class BPEModel(NamedTuple):
    merges: dict          # (a, b) -> new id, in training order
    vocab: dict           # id -> the bytes it stands for
    cache: dict           # word bytes -> ids, filled by encode


FIRST_MERGE_ID = 257
MIN_COUNT = 2


def _merge_pair(seq: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    a, b = pair
    out, i, n = [], 0, len(seq)
    while i < n:
        if i + 1 < n and seq[i] == a and seq[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return out


def train_bpe(word_counts: dict[bytes, int], vocab_size: int = VOCAB_SIZE) -> BPEModel:
    """Byte-level BPE over words with frequencies. Highest count wins; ties go to the smaller pair."""
    words = [(list(w), c) for w, c in word_counts.items()]
    merges: dict = {}
    for k in range(vocab_size - FIRST_MERGE_ID):
        stats: dict = {}
        for seq, count in words:
            for pair in zip(seq, seq[1:]):
                stats[pair] = stats.get(pair, 0) + count
        if not stats:
            break
        top = max(stats.values())
        if top < MIN_COUNT:
            break
        pair = min(p for p, n in stats.items() if n == top)
        merges[pair] = FIRST_MERGE_ID + k
        words = [(_merge_pair(seq, pair, FIRST_MERGE_ID + k), count) for seq, count in words]
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), new_id in merges.items():
        vocab[new_id] = vocab[a] + vocab[b]
    return BPEModel(merges, vocab, {})


def encode(text: str, model: BPEModel) -> list[int]:
    """Token ids for `text`: merges applied in training order inside each word; a whitespace
    byte is emitted as its own id."""
    out, word = [], []

    def flush():
        if word:
            key = bytes(word)
            if key not in model.cache:
                ids = list(key)
                for pair, new_id in model.merges.items():
                    if len(ids) < 2:
                        break
                    ids = _merge_pair(ids, pair, new_id)
                model.cache[key] = ids
            out.extend(model.cache[key])
            word.clear()

    for byte in text.encode("utf-8"):
        if byte in _WHITESPACE:
            flush()
            out.append(byte)
        else:
            word.append(byte)
    flush()
    return out


def count_tokens(model: BPEModel, text: str) -> int:
    """Tokens in `text`, not counting the whitespace ids."""
    return sum(1 for token in encode(text, model) if token not in _WHITESPACE)


def train_on_mixture(share: dict[str, float], n_words: int = TRAIN_WORDS, seed: int = SEED) -> BPEModel:
    """Train a tokenizer on `n_words` words split across the languages by `share`."""
    counts: dict[bytes, int] = {}
    for i, lang in enumerate(LANGS):
        for c in sample_concepts(int(n_words * share[lang]), seed + 100 + i):
            key = LEXICONS[lang][int(c)].encode("utf-8")
            counts[key] = counts.get(key, 0) + 1
    return train_bpe(counts)


PASSAGES = {lang: render(lang, sample_concepts(PASSAGE_WORDS, SEED + 7)) for lang in LANGS}
TOKENIZER = train_on_mixture(TRAIN_SHARE)

if __name__ == "__main__":
    print(f"trained {len(TOKENIZER.merges)} merges on {TRAIN_WORDS} words; the mixture was")
    print("  " + ", ".join(f"{lang} {TRAIN_SHARE[lang]:.0%}" for lang in LANGS))
    _lossless = all(
        b"".join(TOKENIZER.vocab[t] for t in encode(text, TOKENIZER)).decode("utf-8") == text
        for text in PASSAGES.values())
    print(f"encode then expand through the vocabulary gives the original text back, in all six "
          f"scripts: {_lossless}")

## 3. Exercise 1 — `fertility`

**Token fertility** is tokens per unit of text: the rate at which a tokenizer fragments a
language. Two units let you compare languages, because both stay fixed when you translate:
**words** (here, item *k* is the same word in every language) and **characters**. A third,
**bytes**, is a trap. UTF-8 already spends a different number of bytes on each script before
any tokenizer runs, so a per-byte figure mixes the encoding's cost into the tokenizer's: on a
tokenizer with no merges every script scores exactly one token per byte, whatever the tax.

Count tokens with `count_tokens` (it leaves out whitespace). A word is a whitespace-separated
piece. A character is a non-whitespace character, so numerator and denominator leave out the
same spaces. Say which unit you used: the unit goes back in the result.

<details><summary>💡 Hint 1 — what to think about</summary>

The function is asked for a *unit*. What are the two denominators, and which quantity that
you can get from a string in one call (`len` of what?) is deliberately not one of them? Also
ask what should happen when the unit is something else, or when there is nothing to divide by.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Count the tokens once. Choose the denominator by the unit: the number of pieces `str.split()`
gives for words, the number of characters that are not whitespace for chars. Refuse any other
unit and a zero denominator with a `ValueError` that says what was wrong. Return the four
documented keys, with the unit echoed as given.
</details>

In [ ]:
def fertility(model: BPEModel, text: str, unit: str) -> dict:
    """Tokens per `unit` of `text`, where `unit` is "word" or "char".

    Returns a dict with exactly these keys:
      "unit"         the unit you were asked for, echoed back
      "tokens"       count_tokens(model, text)
      "denominator"  the number of words (whitespace-separated pieces) or of characters
                     (non-whitespace characters, matching what count_tokens leaves out)
      "value"        tokens / denominator

    Raise ValueError for any other unit, "byte" included (a byte count differs by script
    before any tokenizer runs), and for a text with nothing to divide by.

    Example:
        >>> m = train_bpe({b"ab": 5}, 258)          # one merge: a + b
        >>> fertility(m, "ab ab ba", "word")
        {'unit': 'word', 'tokens': 4, 'denominator': 3, 'value': 1.3333333333333333}
        >>> fertility(m, "ab ab ba", "char")["denominator"]
        6
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fertility() -> None:
    toy = train_bpe({b"ab": 5}, 258)
    got = fertility(toy, "ab ab ba", "word")
    assert got == {"unit": "word", "tokens": 4, "denominator": 3, "value": 4 / 3}, (
        f"fertility(toy, 'ab ab ba', 'word') gave {got}; expected 4 tokens over 3 words. "
        "Return exactly the four documented keys, with the unit echoed back."
    )
    assert fertility(toy, "ab ab ba", "char")["denominator"] == 6, (
        "the character denominator counts non-whitespace characters: 'ab ab ba' has 6, "
        "not 8. count_tokens leaves the spaces out, so the denominator must too."
    )
    han = PASSAGES["hani"]
    assert fertility(TOKENIZER, han, "char")["denominator"] == len(han.replace(" ", "")), (
        "for Han text a character is three UTF-8 bytes; if the denominator is len(text.encode()) "
        "you have divided by bytes, which hides the tax. Count characters."
    )
    for bad in ("byte", "bytes", "sentence", ""):
        try:
            fertility(TOKENIZER, "ab", bad)
        except ValueError:
            continue
        raise AssertionError(f"unit {bad!r} must raise ValueError; only 'word' and 'char' compare languages")
    try:
        fertility(TOKENIZER, "   ", "word")
    except ValueError:
        pass
    else:
        raise AssertionError("a text with no words has nothing to divide by: raise ValueError")
    print("exercise 1 looks right: words and characters, the unit echoed back, bytes refused")

In [ ]:
if __name__ == "__main__":
    _try("exercise 1", _check_fertility)

Now the tax. Every language below writes the same concepts, word for word, so **tokens** is the honest
cost of the same content and the `vs latn-a` column is that cost as a multiple of the
reference language's. Read the three tokens-per-unit columns against it: one of them is the
cost in disguise, one is comparable only between scripts that pack a word into a similar
number of characters, and one is not a usable comparison at all. Say which is which before
you read the summary lines under the table.

In [ ]:
def fertility_table(model: BPEModel, passages: dict[str, str], ref: str = "latn-a") -> None:
    """Print words, characters, bytes, tokens and the three fertilities. Given to you."""
    base = count_tokens(model, passages[ref])
    print(f"{'language':8} {'words':>6} {'chars':>6} {'bytes':>6} {'tokens':>7} "
          f"{'tok/word':>9} {'tok/char':>9} {'tok/byte':>9} {'vs ' + ref:>10}")
    for lang, text in passages.items():
        w = fertility(model, text, "word")
        c = fertility(model, text, "char")
        n_bytes = len(text.replace(" ", "").encode("utf-8"))
        print(f"{lang:8} {w['denominator']:6d} {c['denominator']:6d} {n_bytes:6d} {w['tokens']:7d} "
              f"{w['value']:9.2f} {c['value']:9.2f} {w['tokens'] / n_bytes:9.2f} "
              f"{w['tokens'] / base:10.2f}")


def _show_fertility() -> None:
    fertility_table(TOKENIZER, PASSAGES)
    tokens = {lang: count_tokens(TOKENIZER, text) for lang, text in PASSAGES.items()}
    worst = max(tokens, key=tokens.get)
    print(f"\nthe same {PASSAGE_WORDS} concepts cost {tokens[worst]} tokens in {worst} and "
          f"{tokens['latn-a']} in latn-a.")
    per_byte = {lang: tokens[lang] / len(PASSAGES[lang].replace(" ", "").encode("utf-8"))
                for lang in LANGS}
    by_tokens, by_byte = sorted(LANGS, key=tokens.get), sorted(LANGS, key=per_byte.get)
    print(f"per byte the spread is {min(per_byte.values()):.2f} to {max(per_byte.values()):.2f}, and it ranks "
          f"{by_byte[0]} cheapest where tokens rank {by_tokens[0]} cheapest: "
          + ("the two orders disagree, so the per-byte figure does not rank these languages by cost."
             if by_byte != by_tokens else "here the two orders happen to agree."))

In [ ]:
if __name__ == "__main__":
    _try("the tax table", _show_fertility, needs=("exercise 1",))

### The tax follows the training mixture

Nothing in the algorithm knows which script it is fragmenting; it merges what it counted most.
So much of the tax follows the mixture the tokenizer saw; what an equal mixture leaves behind
comes from the scripts themselves. Train the same code on an equal mixture and re-measure the
same passages. Petrov et al. and Ahia et al. document this disparity for
production tokenizers (`claims.yaml`); here you can see the mechanism with the answer known.

In [ ]:
def _show_mixture_effect() -> None:
    equal = train_on_mixture({lang: 1 / len(LANGS) for lang in LANGS})
    print(f"{'language':8} {'tok/word skewed':>16} {'tok/word equal':>15}")
    for lang in LANGS:
        a = fertility(TOKENIZER, PASSAGES[lang], "word")["value"]
        b = fertility(equal, PASSAGES[lang], "word")["value"]
        print(f"{lang:8} {a:16.2f} {b:15.2f}")
    spread = lambda m: max(fertility(m, t, "word")["value"] for t in PASSAGES.values()) / min(
        fertility(m, t, "word")["value"] for t in PASSAGES.values())
    print(f"\nbest-to-worst fertility ratio: {spread(TOKENIZER):.2f} skewed, {spread(equal):.2f} equal.")
    a, b = fertility(TOKENIZER, PASSAGES["latn-a"], "word")["value"], fertility(equal, PASSAGES["latn-a"], "word")["value"]
    print(f"The equal mixture is not free: latn-a goes from {a:.2f} to {b:.2f} tokens per word, because the")
    print("vocabulary is a fixed budget and every slot spent on one language is not spent on another.")
    print("Same algorithm, same vocabulary budget, same text to encode: only the mixture changed.")

In [ ]:
if __name__ == "__main__":
    _try("the mixture effect", _show_mixture_effect, needs=("exercise 1",))

## 4. The benchmark and the two systems (given)

The benchmark is four-option multiple choice, so **chance is 0.25**. Each concept exists
twice in each language: a **native** item, written in that language, and a **translated**
item, machine-translated from another language. Relying on machine-translated evaluation sets
is common practice (`claims.yaml`), so "the benchmark" below means the translated items, and
the native items are the control you only get by building them.

Two systems, `A` and `B`, answer every item. Each has a *planted* probability of being right
in each language, and system `A` gets a bonus on translated items: it has, so to speak, seen
translationese. The cell after this one defines those planted numbers; resist reading them until
the last section, which prints them beside what you measured. They were chosen so that every
trap in this lesson shows at these benchmark sizes. A real evaluation makes no such promise:
there, whether a trap bites is exactly what your instruments have to find out.

Items share difficulty: a hard concept is hard for both systems and in both its versions with
probability `SHARED_HARDNESS`, so the systems' answers on one item are correlated. That is
exactly the situation a *paired* comparison is built for, and it is why it is sharper than an
unpaired one here.

In [ ]:
N_CONCEPTS = {"latn-a": 600, "cyrl": 300, "hani": 240, "deva": 120, "arab": 80, "latn-b": 30}
P_NATIVE = {   # planted P(correct) on natively written items, per system and language
    "A": {"latn-a": 0.86, "cyrl": 0.74, "hani": 0.70, "deva": 0.35, "arab": 0.28, "latn-b": 0.25},
    "B": {"latn-a": 0.78, "cyrl": 0.68, "hani": 0.66, "deva": 0.62, "arab": 0.60, "latn-b": 0.58},
}
TRANS_BONUS = {"A": 0.05, "B": 0.01}   # planted extra P(correct) on translated items
SHARED_HARDNESS = 0.6
CHANCE = 0.25                          # four options


class Item(NamedTuple):
    item_id: int
    lang: str
    origin: str      # "native" or "translated"
    concept: int     # the native and the translated version of one concept share this


def true_p(system: str, lang: str, origin: str) -> float:
    """The planted probability that `system` answers an `origin` item in `lang` correctly."""
    return P_NATIVE[system][lang] + (TRANS_BONUS[system] if origin == "translated" else 0.0)


def make_benchmark(seed: int = SEED) -> tuple[list[Item], dict[str, dict[int, int]]]:
    """Items in a shuffled order, and each system's 0/1 result keyed by item id."""
    rng = np.random.default_rng(seed)
    rows, concept = [], 0
    for lang in LANGS:
        for _ in range(N_CONCEPTS[lang]):
            rows.append((lang, "native", concept))
            rows.append((lang, "translated", concept))
            concept += 1
    order = rng.permutation(len(rows))
    items = [Item(i, *rows[j]) for i, j in enumerate(order)]
    shared_level = rng.random(concept)
    is_shared = rng.random(concept) < SHARED_HARDNESS
    own_level = {s: rng.random(len(items)) for s in ("A", "B")}
    results: dict[str, dict[int, int]] = {}
    for s in ("A", "B"):
        results[s] = {
            it.item_id: int((shared_level[it.concept] if is_shared[it.concept]
                             else own_level[s][it.item_id]) < true_p(s, it.lang, it.origin))
            for it in items}
    return items, results


def subset(items: list[Item], origin: str | None = None, lang: str | None = None) -> list[Item]:
    """The items with this origin and/or language (None means any)."""
    return [it for it in items if (origin is None or it.origin == origin)
            and (lang is None or it.lang == lang)]


def outcomes(results: dict[int, int], items: list[Item]) -> np.ndarray:
    """One system's 0/1 outcomes on `items`, as an integer array in the order given."""
    return np.array([results[it.item_id] for it in items], dtype=np.int64)


ITEMS, RESULTS = make_benchmark()
TRANSLATED = subset(ITEMS, origin="translated")     # "the published benchmark"
NATIVE = subset(ITEMS, origin="native")             # the control you have to build yourself

if __name__ == "__main__":
    print(f"{'language':8} {'concepts':>9} {'items':>6}   (each concept: one native + one translated item)")
    for lang in LANGS:
        print(f"{lang:8} {N_CONCEPTS[lang]:9d} {len(subset(ITEMS, lang=lang)):6d}")
    print(f"\n{len(ITEMS)} items in all, shuffled: an item's position tells you nothing about its language.")
    print("The published benchmark is the translated half, which is what the next sections evaluate.")

## 5. Exercise 2 — cost per correct answer, and what fits in the window

A commercial API meters usage in tokens (`claims.yaml`), and a model's context window is a
fixed number of tokens. Two consequences follow, and you will compute both.

* **Cost per correct answer**: the money spent divided by the answers that were right. It
  multiplies two taxes: a language that costs more tokens per question *and* gets more of them
  wrong pays for the wrong answers as well as the extra tokens. (`usd_per_mtok` is a price per
  *million* tokens.) A system that got nothing right has no finite cost per correct answer.
* **Units in the window**: how many whole words (or characters) fit in a fixed window at a given
  fertility.

<details><summary>💡 Hint 1 — what to think about</summary>

For the cost: what is the total number of tokens billed, what does one million of them cost,
and what are you dividing by, correct answers or items? For the window: is the answer a whole
number, and what should a non-positive fertility do?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Total tokens are tokens-per-item times the number of items. Convert to dollars with the per-
million price, then divide by the number of correct answers, returning infinity when there are
none. For the window, divide the window by the tokens per unit and round *down*, because half
a word does not fit; refuse a fertility that is zero or negative.
</details>

In [ ]:
def cost_per_correct(tokens_per_item: float, n_items: int, n_correct: int,
                     usd_per_mtok: float) -> float:
    """USD spent in total, divided by the number of correct answers.

    `n_items` items of `tokens_per_item` tokens are billed at `usd_per_mtok` dollars per MILLION
    tokens. If `n_correct` is 0 return `float("inf")`.

    Example:
        >>> cost_per_correct(100, 1000, 500, 2.0)   # 100,000 tokens = $0.20, over 500 right
        0.0004
        >>> cost_per_correct(100, 1000, 0, 2.0)
        inf
    """
    # YOUR CODE HERE
    raise NotImplementedError


def units_in_window(window_tokens: int, fertility_value: float) -> int:
    """How many whole words (or characters) fit in `window_tokens` tokens at `fertility_value`
    tokens per unit. Round down. Raise ValueError if `fertility_value` is not positive.

    Example:
        >>> units_in_window(8192, 2.0)
        4096
        >>> units_in_window(11, 4.0)      # 2.75 units fit, so 2
        2
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cost_and_window() -> None:
    got = cost_per_correct(100, 1000, 500, 2.0)
    assert abs(got - 0.0004) < 1e-12, (
        f"cost_per_correct(100, 1000, 500, 2.0) gave {got}, expected 0.0004: 100,000 tokens at "
        "$2 per MILLION is $0.20, and that is shared over the 500 CORRECT answers. Check for "
        "a missing 1e6, or a division by items where it should be by correct answers."
    )
    assert cost_per_correct(50, 10, 0, 1.0) == float("inf"), (
        "with no correct answer the cost per correct answer is unbounded: return float('inf'), "
        "do not divide by zero or return 0."
    )
    assert cost_per_correct(200, 400, 40, 1.0) == 2 * cost_per_correct(100, 400, 40, 1.0) or abs(
        cost_per_correct(200, 400, 40, 1.0) - 2 * cost_per_correct(100, 400, 40, 1.0)) < 1e-12, (
        "twice the tokens per item must cost twice as much per correct answer"
    )
    assert units_in_window(11, 4.0) == 2 and units_in_window(8192, 2.0) == 4096, (
        "units_in_window rounds DOWN to whole units: 11 tokens at 4 per unit fits 2, not 3"
    )
    try:
        units_in_window(100, 0.0)
    except ValueError:
        pass
    else:
        raise AssertionError("a fertility of 0 must raise ValueError, not divide by zero or return inf")
    print("exercise 2 looks right: cost is per correct answer, and the window rounds down")

In [ ]:
if __name__ == "__main__":
    _try("exercise 2", _check_cost_and_window)

Now price the benchmark. Every item is a question of `WORDS_PER_ITEM` words. A word costs the
language's measured tokens per word, so one item costs `WORDS_PER_ITEM` times that. The price
and the window below are round illustrative numbers, not any vendor's. System `A`'s answers on
the translated benchmark supply the accuracy.

Two ratios are printed next to the cost so you can see it multiply: the *token* ratio (this
language's tokens per item over `latn-a`'s) and the *accuracy* ratio (`latn-a`'s accuracy over
this language's).

In [ ]:
WORDS_PER_ITEM = 60       # words in one benchmark question (planted, the same in every language)
USD_PER_MTOK = 2.0        # dollars per million tokens: an illustrative round number
WINDOW_TOKENS = 8192      # a round-number context window


def _show_cost() -> None:
    rows = {}
    for lang in LANGS:
        w = fertility(TOKENIZER, PASSAGES[lang], "word")["value"]
        c = fertility(TOKENIZER, PASSAGES[lang], "char")["value"]
        items = subset(TRANSLATED, lang=lang)
        correct = int(outcomes(RESULTS["A"], items).sum())
        tokens_per_item = WORDS_PER_ITEM * w
        rows[lang] = dict(acc=correct / len(items), tpi=tokens_per_item,
                          cost=cost_per_correct(tokens_per_item, len(items), correct, USD_PER_MTOK),
                          words=units_in_window(WINDOW_TOKENS, w), chars=units_in_window(WINDOW_TOKENS, c))
    ref = rows["latn-a"]
    print(f"system A on the translated benchmark; price ${USD_PER_MTOK:.2f} per million tokens\n")
    print(f"{'language':8} {'acc':>6} {'tok/item':>9} {'USD/1000 right':>15} {'cost ratio':>11} "
          f"{'= tokens':>9} {'x 1/acc':>8} {'words fit':>10} {'chars fit':>10}")
    for lang, r in rows.items():
        print(f"{lang:8} {r['acc']:6.3f} {r['tpi']:9.1f} {1000 * r['cost']:15.4f} "
              f"{r['cost'] / ref['cost']:11.2f} {r['tpi'] / ref['tpi']:9.2f} {ref['acc'] / r['acc']:8.2f} "
              f"{r['words']:10d} {r['chars']:10d}")
    print("\nThe cost ratio is the token ratio times the accuracy ratio: the tokenizer's tax and the")
    print("benchmark's accuracy gap compound. The window fits fewer words in a high-fertility")
    print("language; its character count is not comparable across scripts, for the reason above.")

In [ ]:
if __name__ == "__main__":
    _try("the price of a correct answer", _show_cost,
         needs=("exercise 1", "exercise 2"))

## 6. Exercise 3 — a bootstrap interval over items

Each accuracy above is a mean of 0/1 outcomes over a finite sample of items, so it carries
sampling error, and a small language carries more. A **percentile bootstrap** estimates it:
draw `n` items *with replacement* from the `n` you have, take the mean, do it `n_boot` times,
and report the middle `1 - alpha` of those means. The **unit resampled is the item**: that is
what was sampled from the population of questions, and it is what a language's interval is
about.

Use `np.random.default_rng(seed)` and draw the indices in one call, so the interval is
reproducible.

<details><summary>💡 Hint 1 — what to think about</summary>

What has to stay the same in every resample, and what varies? Where does the `n` in the
interval's width come from: the number of items, or the number of resamples? And what should
the interval do when every value is identical?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Make an array of the values. Draw a matrix of random item indices with `n_boot` rows and `n`
columns, each entry between 0 and n-1. Index the values with it, take each row's mean, then
read off the `alpha/2` and `1 - alpha/2` percentiles (as percentages) of those means. Return
the mean of the *original* values with the two ends. An empty input is an error.
</details>

In [ ]:
N_BOOT = 2000


def bootstrap_ci(values, n_boot: int = N_BOOT, seed: int = 0,
                 alpha: float = 0.05) -> tuple[float, float, float]:
    """`(mean, lo, hi)`: the mean of `values` and its percentile-bootstrap interval, resampling items.

    `values` is a sequence of numbers (here 0/1 outcomes, one per item). Raise ValueError if it
    is empty. The same `seed` must give the same interval.

    Example:
        >>> bootstrap_ci([1, 1, 1, 1], seed=0)
        (1.0, 1.0, 1.0)
        >>> mean, lo, hi = bootstrap_ci([1, 0] * 50, seed=0)
        >>> mean, lo < mean < hi
        (0.5, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bootstrap_ci() -> None:
    assert bootstrap_ci([1, 1, 1, 1], seed=0) == (1.0, 1.0, 1.0), (
        "identical values must give a zero-width interval at that value; every resample of four "
        "ones has mean 1"
    )
    values = [1, 0, 0, 1] * 50
    mean, lo, hi = bootstrap_ci(values, seed=1)
    assert mean == 0.5 and lo < 0.5 < hi, (
        f"got mean {mean} and interval [{lo}, {hi}] for 200 fair-coin outcomes: the mean is the "
        "original data's, and the interval must straddle it"
    )
    assert bootstrap_ci(values, seed=1) == (mean, lo, hi), "the same seed must give the same interval"
    width = hi - lo
    assert 0.09 < width < 0.16, (
        f"interval width {width:.3f} for n=200 items at p=0.5; the standard error there is about "
        "0.035, so a 95% interval is about 0.14 wide. A much wider interval usually means the "
        "spread of the values was used where the spread of the MEAN belongs; a much narrower one, "
        "resamples of the wrong size."
    )
    print(f"exercise 3 looks right: 200 items at p=0.5 gave [{lo:.3f}, {hi:.3f}]")

In [ ]:
if __name__ == "__main__":
    _try("exercise 3", _check_bootstrap_ci)

Apply it to every language, for both systems, on the translated benchmark. Watch the width
follow the item counts: the same accuracy is a much weaker statement in `latn-b` than in
`latn-a`. An asterisk marks an interval whose lower end is at or below chance.

In [ ]:
def language_intervals(system: str, items: list[Item]) -> dict[str, tuple[float, float, float]]:
    """`{lang: (accuracy, lo, hi)}` from `bootstrap_ci` over each language's items. Given to you."""
    return {lang: bootstrap_ci(outcomes(RESULTS[system], subset(items, lang=lang)), seed=SEED)
            for lang in LANGS}


def _show_language_intervals() -> None:
    a, b = language_intervals("A", TRANSLATED), language_intervals("B", TRANSLATED)
    print(f"{'language':8} {'items':>6}   {'system A':^24}   {'system B':^24}")
    for lang in LANGS:
        row = []
        for m, lo, hi in (a[lang], b[lang]):
            row.append(f"{m:.3f} [{lo:.3f}, {hi:.3f}]" + ("*" if lo <= CHANCE else " "))
        print(f"{lang:8} {len(subset(TRANSLATED, lang=lang)):6d}   {row[0]:^24}   {row[1]:^24}")
    print(f"\n* = the interval's lower end is at or below chance ({CHANCE}).")

In [ ]:
if __name__ == "__main__":
    _try("per-language intervals", _show_language_intervals, needs=("exercise 3",))

## 7. Exercise 4 — an interval for the macro-average

A leaderboard usually reports one number for the whole benchmark. The **macro-average** is the
unweighted mean of the per-language accuracies, so the smallest language counts as much as the
largest. Its interval has to be built the same way the number is: each language's
accuracy is a mean over *its items*, and it is those items that are the random sample. So
**resample items inside each language**, recompute every language's accuracy, average them,
and repeat. The tempting shortcut resamples the six language accuracies themselves; the
next cell shows what that interval measures.

<details><summary>💡 Hint 1 — what to think about</summary>

What are the random draws in this evaluation: languages, or questions? If you keep the
languages fixed and let only the items vary, what changes from one resample to the next?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Use one generator. For each language, draw an index matrix of `n_boot` by *that language's*
item count, take the row means of its outcomes, and add it to a running total. Divide the
total by the number of languages to get `n_boot` macro-averages, then take the same two
percentiles as before. The reported centre is the macro-average of the original data.
</details>

In [ ]:
def macro_ci(outcomes_by_lang: dict[str, np.ndarray], n_boot: int = N_BOOT, seed: int = 0,
             alpha: float = 0.05) -> tuple[float, float, float]:
    """`(macro, lo, hi)`: the unweighted mean of per-language accuracies, with a bootstrap interval
    that resamples ITEMS within each language.

    `outcomes_by_lang` maps a language to its 0/1 outcomes. Raise ValueError if it is empty or a
    language has no outcomes.

    Example:
        >>> macro_ci({"x": np.array([1, 1, 1, 1]), "y": np.array([0, 0])})
        (0.5, 0.5, 0.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_macro_ci() -> None:
    assert macro_ci({"x": np.array([1, 1, 1, 1]), "y": np.array([0, 0])}) == (0.5, 0.5, 0.5), (
        "languages whose outcomes are all identical leave nothing to resample: the interval must "
        "collapse to the macro-average, 0.5 here"
    )
    rng = np.random.default_rng(3)
    data = {name: (rng.random(80) < p).astype(int) for name, p in (("x", 0.9), ("y", 0.5), ("z", 0.3))}
    macro, lo, hi = macro_ci(data, seed=2)
    expect = sum(v.mean() for v in data.values()) / 3
    assert abs(macro - expect) < 1e-12, (
        f"macro {macro} but the mean of the three language accuracies is {expect}: it is an "
        "unweighted mean of accuracies, not correct answers over all items"
    )
    assert 0.06 < hi - lo < 0.13, (
        f"interval width {hi - lo:.3f}. With 80 items in each of three languages the sampling "
        "error of the macro-average is about 0.03, so the width is about 0.11. A width near 0.5 "
        "means the spread BETWEEN languages was resampled; the languages are fixed, the items "
        "are the sample."
    )
    print(f"exercise 4 looks right: macro {macro:.3f} [{lo:.3f}, {hi:.3f}] from three languages")

In [ ]:
if __name__ == "__main__":
    _try("exercise 4", _check_macro_ci)

The wrong interval, run beside the right one. `language_resampled_ci` draws languages with
replacement: its width mostly measures how different the languages are from each other, which
is a fact about the benchmark's design, not about how well its items pin down each accuracy.

In [ ]:
def language_resampled_ci(outcomes_by_lang: dict[str, np.ndarray], n_boot: int = N_BOOT,
                          seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """The WRONG unit, shown so you can see it: resamples the per-language accuracies. Given."""
    accs = np.array([v.mean() for v in outcomes_by_lang.values()])
    idx = np.random.default_rng(seed).integers(0, len(accs), size=(n_boot, len(accs)))
    boots = accs[idx].mean(axis=1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(accs.mean()), float(lo), float(hi)


def _show_macro_ci() -> None:
    print(f"{'system':7} {'macro':>7} {'items resampled':>22} {'width':>7}   {'languages resampled':>22} {'width':>7}")
    for system in ("A", "B"):
        data = {lang: outcomes(RESULTS[system], subset(TRANSLATED, lang=lang)) for lang in LANGS}
        m, lo, hi = macro_ci(data, seed=SEED)
        _, wlo, whi = language_resampled_ci(data, seed=SEED)
        print(f"{system:7} {m:7.3f} {f'[{lo:.3f}, {hi:.3f}]':>22} {hi - lo:7.3f}   "
              f"{f'[{wlo:.3f}, {whi:.3f}]':>22} {whi - wlo:7.3f}")
    print("\nMore items would narrow the first interval, because items are what was sampled. The second")
    print("would not narrow at all: its width is set by how different the six languages are.")

In [ ]:
if __name__ == "__main__":
    _try("the wrong unit", _show_macro_ci, needs=("exercise 4",))

## 8. Exercise 5 — pairing two systems on the same items

To say whether `A` beats `B` you must run them on **the same items**. If each system saw a
different set of questions, a difference in their scores is partly a difference in how hard
the questions were, and no interval can separate the two. So the comparison starts by
**aligning on item id**, never on position: two result lists can have the same length and the
same order and still be about different items, or hold the same items in a different order.

Return the shared ids in ascending order with each system's outcome, and **refuse** if the
two systems did not answer exactly the same ids.

<details><summary>💡 Hint 1 — what to think about</summary>

What has to be true of the two sets of item ids before a per-item comparison means anything?
If you pair the k-th result of each, what are you assuming about their order?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compare the two key sets first. If they differ, count the ids only in the first and only in the
second and raise a `ValueError` whose message gives both counts. Otherwise sort the ids and read
each system's outcome for every id in that order, into two integer arrays.
</details>

In [ ]:
def align_paired(res_a: dict[int, int], res_b: dict[int, int]) -> tuple[list[int], np.ndarray, np.ndarray]:
    """Match two systems' results on item id. Returns `(ids, a, b)`.

    `ids` is ascending; `a[i]` and `b[i]` are the two systems' 0/1 outcomes on `ids[i]`, as
    integer numpy arrays. Raise `ValueError` if the two do not cover exactly the same item ids;
    the message says how many are only in the first and how many only in the second.

    Example:
        >>> align_paired({2: 1, 1: 0}, {1: 1, 2: 1})
        ([1, 2], array([0, 1]), array([1, 1]))
        >>> align_paired({1: 0}, {1: 0, 2: 1})
        Traceback (most recent call last):
        ValueError: 0 item ids only in the first system, 1 only in the second
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_align_paired() -> None:
    ids, a, b = align_paired({5: 1, 3: 0, 4: 1}, {4: 0, 5: 1, 3: 1})
    assert ids == [3, 4, 5] and a.tolist() == [0, 1, 1] and b.tolist() == [1, 0, 1], (
        f"got ids {ids}, a {a.tolist()}, b {b.tolist()}: align on item id and sort the ids; "
        "the two dicts list the same items in different orders, so pairing by position is wrong"
    )
    for first, second in (({1: 0, 2: 1}, {1: 0, 2: 1, 3: 1}), ({1: 0, 2: 1}, {1: 0, 3: 1}),
                          ({1: 0, 2: 1, 3: 1}, {1: 0, 2: 1})):
        try:
            align_paired(first, second)
        except ValueError:
            continue
        raise AssertionError(
            f"{sorted(first)} against {sorted(second)} must raise ValueError: the two systems did not "
            "answer the same items, so a paired comparison is not defined. Zipping the shorter "
            "one, or intersecting silently, hides exactly the mistake this function exists to stop."
        )
    print("exercise 5 looks right: aligned on id, and refused when the item sets differ")

In [ ]:
if __name__ == "__main__":
    _try("exercise 5", _check_align_paired)

## 9. Exercise 6 — a paired-bootstrap interval for the difference

Now the comparison itself. With the two outcome arrays aligned, take the per-item difference
`a - b` (each entry is -1, 0 or +1), and bootstrap it: draw items with replacement and recompute
the mean difference. Because the **same resampled items feed both systems**, the correlation
between their answers cancels out of the interval, so a paired comparison is sharper than an
unpaired one whenever the systems tend to fail on the same items. (Koehn 2004, in
`claims.yaml`, describes bootstrap resampling for exactly this significance question.) The
unpaired shortcut resamples each system independently; it is provided below so you can put the
two side by side.

<details><summary>💡 Hint 1 — what to think about</summary>

If you resample system A's items and system B's items *separately*, what has happened to the
fact that item 17 was hard for both? What single quantity per item carries the pairing?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Call your `align_paired`, subtract the arrays to get one difference per item, and hand that
array to the same resampling as exercise 3: one index matrix, one row-mean per resample, two
percentiles. The reported centre is the mean difference of the original data, first system
minus second.
</details>

In [ ]:
def paired_diff_ci(res_a: dict[int, int], res_b: dict[int, int], n_boot: int = N_BOOT,
                   seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """`(diff, lo, hi)`: mean of `a - b` per item, with a paired percentile-bootstrap interval.

    Aligns the two results with your `align_paired` (so mismatched items raise ValueError), then
    resamples items once and applies the same resample to both systems.

    Example:
        >>> a = {i: 1 for i in range(100)}
        >>> b = {i: 1 - (i % 10 == 0) for i in range(100)}     # b misses every tenth item
        >>> d, lo, hi = paired_diff_ci(a, b, seed=0)
        >>> round(d, 2), lo > 0
        (0.1, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_paired_diff_ci() -> None:
    a = {i: 1 for i in range(100)}
    b = {i: 1 - (i % 10 == 0) for i in range(100)}
    d, lo, hi = paired_diff_ci(a, b, seed=0)
    assert abs(d - 0.1) < 1e-12 and lo > 0, (
        f"got diff {d} with interval [{lo}, {hi}]. B misses exactly ten items that A gets, so the "
        "paired difference is 0.10 and the interval must sit above zero: resample the per-item "
        "DIFFERENCES, not the two systems separately"
    )
    x = {i: int(i % 3 == 0) for i in range(90)}
    y = {i: int(i % 3 == 0) for i in range(89)}
    y[200] = 1
    try:
        paired_diff_ci(x, y)
    except ValueError:
        pass
    else:
        raise AssertionError("two systems evaluated on different item ids must raise ValueError, "
                             "not be compared anyway (call align_paired)")
    print(f"exercise 6 looks right: diff {d:.2f} with paired interval [{lo:.3f}, {hi:.3f}]")

In [ ]:
if __name__ == "__main__":
    _try("exercise 6", _check_paired_diff_ci, needs=("exercise 5",))

Run it on the whole translated benchmark, beside the unpaired interval for the same gap. Then
run it the way a hurried evaluation often does: system `B` scored only on the first three
(largest) languages, `A` on everything. Your `align_paired` should refuse, and the message is the
point.

In [ ]:
def unpaired_diff_ci(a: np.ndarray, b: np.ndarray, n_boot: int = N_BOOT, seed: int = 0,
                     alpha: float = 0.05) -> tuple[float, float, float]:
    """The unpaired shortcut, shown so you can see it: A and B are resampled independently."""
    rng = np.random.default_rng(seed)
    ia = rng.integers(0, len(a), size=(n_boot, len(a)))
    ib = rng.integers(0, len(b), size=(n_boot, len(b)))
    boots = a[ia].mean(axis=1) - b[ib].mean(axis=1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(a.mean() - b.mean()), float(lo), float(hi)


def _show_paired() -> None:
    res_a = {it.item_id: RESULTS["A"][it.item_id] for it in TRANSLATED}
    res_b = {it.item_id: RESULTS["B"][it.item_id] for it in TRANSLATED}
    d, lo, hi = paired_diff_ci(res_a, res_b, seed=SEED)
    ud, ulo, uhi = unpaired_diff_ci(outcomes(RESULTS["A"], TRANSLATED),
                                    outcomes(RESULTS["B"], TRANSLATED), seed=SEED)
    print("A minus B, micro accuracy over every translated item")
    print(f"  paired   {d:+.3f}  [{lo:+.3f}, {hi:+.3f}]  width {hi - lo:.3f}")
    print(f"  unpaired {ud:+.3f}  [{ulo:+.3f}, {uhi:+.3f}]  width {uhi - ulo:.3f}")
    big_three = {it.item_id for it in TRANSLATED if it.lang in LANGS[:3]}
    partial_b = {i: r for i, r in res_b.items() if i in big_three}
    print("\nnow B is scored on the three largest languages only, and A on everything:")
    try:
        paired_diff_ci(res_a, partial_b, seed=SEED)
    except ValueError as exc:
        print(f"  refused: {exc}")
    same_a = {i: r for i, r in res_a.items() if i in big_three}
    dd, dlo, dhi = paired_diff_ci(same_a, partial_b, seed=SEED)
    print(f"  the same three languages for both: {dd:+.3f}  [{dlo:+.3f}, {dhi:+.3f}]")
    naive = float(np.mean(list(res_a.values())) - np.mean(list(partial_b.values())))
    print(f"  the naive difference of the two headline numbers: {naive:+.3f}, which mixes two item sets")

In [ ]:
if __name__ == "__main__":
    _try("paired against unpaired", _show_paired, needs=("exercise 5", "exercise 6"))

## 10. Exercise 7 — macro and micro averages

One number per system hides a choice. The **micro-average** pools every item: total correct
over total items, so a language counts in proportion to its share of the benchmark. The
**macro-average** averages the per-language accuracies with equal weight, so the smallest
language counts as much as the largest (Manning et al., in `claims.yaml`, define both). When
the systems' strengths are in different languages the two can crown different winners.

Take per-language `(n_correct, n_items)` pairs and return both averages. Skip a language with
zero items in both.

<details><summary>💡 Hint 1 — what to think about</summary>

In which of the two averages does the number of items in a language matter, and in which
does it disappear once the language's accuracy is computed? What breaks for a language that
has no items?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Drop languages whose item count is zero (and raise `ValueError` if none are left). The macro is
the plain mean of each remaining language's `correct / items`. The micro is the sum of the
`correct` values over the sum of the `items`. Return a dict with the keys `"macro"` and `"micro"`.
</details>

In [ ]:
def aggregate(counts: dict[str, tuple[int, int]]) -> dict[str, float]:
    """`{"macro": ..., "micro": ...}` from `{language: (n_correct, n_items)}`.

    macro is the unweighted mean of each language's accuracy; micro is total correct over total
    items. Languages with `n_items == 0` are skipped in both; raise `ValueError` if every
    language is skipped (or `counts` is empty).

    Example:
        >>> aggregate({"big": (90, 100), "small": (1, 10)})
        {'macro': 0.5, 'micro': 0.8272727272727273}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_aggregate() -> None:
    got = aggregate({"big": (90, 100), "small": (1, 10)})
    assert abs(got["macro"] - 0.5) < 1e-12 and abs(got["micro"] - 91 / 110) < 1e-12, (
        f"got {got}. Macro is the mean of the two accuracies (0.9 and 0.1); micro is 91 correct over "
        "110 items. If both numbers agree you have computed the same average twice."
    )
    same = aggregate({"a": (5, 10), "b": (50, 100)})
    assert abs(same["macro"] - 0.5) < 1e-12 and abs(same["micro"] - 0.5) < 1e-12, (
        "equal accuracies must give the same number either way"
    )
    skipped = aggregate({"a": (4, 8), "empty": (0, 0)})
    assert abs(skipped["macro"] - 0.5) < 1e-12, (
        "a language with no items has no accuracy: skip it, do not count it as 0 and do not divide by zero"
    )
    try:
        aggregate({"empty": (0, 0)})
    except ValueError:
        pass
    else:
        raise AssertionError("with no language left there is nothing to average: raise ValueError")
    print("exercise 7 looks right: macro weights languages equally, micro weights items equally")

In [ ]:
if __name__ == "__main__":
    _try("exercise 7", _check_aggregate)

Rank the two systems by each average, on the translated benchmark and then on the native
control. `rank_systems` (given) sorts best first, breaking ties by name. The first column of
the item-weights table is why micro follows the large languages.

In [ ]:
def language_counts(system: str, items: list[Item]) -> dict[str, tuple[int, int]]:
    """`{lang: (n_correct, n_items)}` for one system on `items`. Given to you."""
    out = {}
    for lang in LANGS:
        sel = subset(items, lang=lang)
        out[lang] = (int(outcomes(RESULTS[system], sel).sum()), len(sel))
    return out


def rank_systems(counts_by_system: dict[str, dict[str, tuple[int, int]]], how: str) -> list[str]:
    """System names, best first by `aggregate(...)[how]`, ties broken by name. Given to you."""
    scores = {s: aggregate(c)[how] for s, c in counts_by_system.items()}
    return sorted(scores, key=lambda s: (-scores[s], s))


def _show_aggregates() -> None:
    for label, items in (("translated (the published benchmark)", TRANSLATED), ("native (the control)", NATIVE)):
        counts = {s: language_counts(s, items) for s in ("A", "B")}
        agg = {s: aggregate(counts[s]) for s in counts}
        print(label)
        print(f"  {'system':7} {'macro':>7} {'micro':>7}")
        for s in ("A", "B"):
            print(f"  {s:7} {agg[s]['macro']:7.3f} {agg[s]['micro']:7.3f}")
        macro_top, micro_top = rank_systems(counts, "macro")[0], rank_systems(counts, "micro")[0]
        print(f"  first place by macro: {macro_top}; by micro: {micro_top}"
              + ("   <- the aggregation decides the winner" if macro_top != micro_top else ""))
    total = len(TRANSLATED)
    print("\nshare of the translated benchmark's items, against a language's weight in the macro-average:")
    for lang in LANGS:
        print(f"  {lang:7} {len(subset(TRANSLATED, lang=lang)) / total:6.1%} of items   {1 / len(LANGS):6.1%} of macro")

In [ ]:
if __name__ == "__main__":
    _try("macro against micro", _show_aggregates, needs=("exercise 7",))

## 11. Exercise 8 — the language an aggregate hides

A respectable macro-average can contain a language in which the system is barely above guessing.
With few items per language that is not always visible in the point estimate, which can sit a
little above chance while the interval reaches chance and below. The honest report names every
language whose interval **does not clear chance**: its lower end is at or below `chance`.

Take `{language: (accuracy, lo, hi)}` and the chance level and return the languages that
fail to clear it, in the order given.

<details><summary>💡 Hint 1 — what to think about</summary>

Which end of the interval decides whether the language is distinguishable from guessing? Is a
language whose point estimate is above chance, but whose interval is not, safe to leave out?
And what should happen when the lower end is exactly equal to chance?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the languages in the order given and keep each one whose lower end is at or below the chance
level. Ignore the point estimate: an interval is what says whether the data can tell the language
from guessing. Return the kept names as a list.
</details>

In [ ]:
def weak_languages(intervals: dict[str, tuple[float, float, float]], chance: float) -> list[str]:
    """Languages whose interval does not clear `chance`: lower end `<= chance`, in input order.

    `intervals` maps a language to `(accuracy, lo, hi)`.

    Example:
        >>> weak_languages({"a": (0.9, 0.85, 0.95), "b": (0.30, 0.22, 0.40), "c": (0.2, 0.1, 0.3)}, 0.25)
        ['b', 'c']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_weak_languages() -> None:
    fixture = {"a": (0.9, 0.85, 0.95), "b": (0.30, 0.22, 0.40), "c": (0.2, 0.1, 0.3)}
    got = weak_languages(fixture, 0.25)
    assert got == ["b", "c"], (
        f"got {got}, expected ['b', 'c']. 'b' has an accuracy above chance but an interval that "
        "reaches below it: its lower end decides, not its point estimate. 'c' is below chance."
    )
    assert weak_languages({"x": (0.5, 0.4, 0.6)}, 0.25) == [], "a language clearly above chance is not weak"
    print("exercise 8 looks right: an interval that reaches chance is flagged, whatever the point estimate says")

In [ ]:
if __name__ == "__main__":
    _try("exercise 8", _check_weak_languages)

Report system `A` the honest way: the macro-average with its interval, and the list. The
headline number alone would not have mentioned any language.

In [ ]:
def _show_weak() -> None:
    for system in ("A", "B"):
        intervals = language_intervals(system, TRANSLATED)
        data = {lang: outcomes(RESULTS[system], subset(TRANSLATED, lang=lang)) for lang in LANGS}
        m, lo, hi = macro_ci(data, seed=SEED)
        weak = weak_languages(intervals, CHANCE)
        print(f"system {system}: macro {m:.3f} [{lo:.3f}, {hi:.3f}]; "
              + (f"interval reaches chance ({CHANCE}) in: " + ", ".join(
                  f"{lang} ({intervals[lang][0]:.3f}, [{intervals[lang][1]:.3f}, {intervals[lang][2]:.3f}], "
                  f"{len(subset(TRANSLATED, lang=lang))} items)" for lang in weak)
                 if weak else "every language's interval clears chance"))
    if any(weak_languages(language_intervals(s, TRANSLATED), CHANCE) for s in ("A", "B")):
        print("\nThe macro-average alone would not have named those languages: it is compatible with a")
        print("language the benchmark cannot tell from guessing.")

In [ ]:
if __name__ == "__main__":
    _try("what the aggregate hides", _show_weak, needs=("exercise 3", "exercise 4", "exercise 8"))

## 12. Exercise 9 — the translation artefact

Relying on machine-translated evaluation sets is common practice, and translation can
introduce artefacts that distort a question (the Global-MMLU authors); existing cross-lingual
models are sensitive to them (Artetxe et al.; both in `claims.yaml`). Here system `A` is planted to be
rewarded for them. You cannot see that by looking at one benchmark: you need the same
concepts in both forms. Every concept here has a native and a translated item, so the
comparison can be **paired by concept**.

Take the items and one system's results, optionally restricted to one language, and pair each
concept's translated item with its native item. Only concepts that have **both** count:
the gap is measured on the matched concepts alone.

<details><summary>💡 Hint 1 — what to think about</summary>

What identifies "the same question" in a native and a translated version, and is it the item
id? If some concept exists in only one form, should its item move the gap?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Select the items (by language if one is given), and group each concept's native and translated
outcome. Keep concepts that have both, in ascending concept order. Return the number of
such concepts, the two accuracies over them, their difference (translated minus native) and the
per-concept differences as an integer array; raise `ValueError` if no concept has both.
</details>

In [ ]:
def origin_gap(items: list[Item], results: dict[int, int], lang: str | None = None) -> dict:
    """Pair each concept's translated item with its native item and measure the gap.

    Only concepts that have BOTH a native and a translated item among the selected items count
    (`lang=None` selects every language). Returns a dict with exactly these keys:
      "n_pairs"        how many concepts were matched
      "native_acc"     mean native outcome over the matched concepts
      "translated_acc" mean translated outcome over the matched concepts
      "gap"            translated_acc - native_acc
      "diffs"          integer array: translated minus native outcome for each matched concept,
                       in ascending concept order

    Raise `ValueError` if no concept has both versions.

    Example:
        >>> items = [Item(0, "x", "native", 7), Item(1, "x", "translated", 7), Item(2, "x", "translated", 8)]
        >>> r = origin_gap(items, {0: 0, 1: 1, 2: 1}, "x")
        >>> r["n_pairs"], r["gap"], r["diffs"].tolist()
        (1, 1.0, [1])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_origin_gap() -> None:
    items = [Item(0, "x", "native", 7), Item(1, "x", "translated", 7), Item(2, "x", "translated", 8),
             Item(3, "x", "native", 9), Item(4, "x", "translated", 9), Item(5, "y", "native", 20),
             Item(6, "y", "translated", 20)]
    results = {0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 0}
    r = origin_gap(items, results, "x")
    assert set(r) == {"n_pairs", "native_acc", "translated_acc", "gap", "diffs"}, (
        f"result keys are {sorted(r)}; return exactly the five documented keys"
    )
    assert r["n_pairs"] == 2 and r["diffs"].tolist() == [1, 0], (
        f"got {r['n_pairs']} pairs and diffs {r['diffs'].tolist()}; concept 8 has only a translated "
        "item, so it is unmatched and must not count. Pair by concept, in ascending order."
    )
    assert abs(r["native_acc"] - 0.5) < 1e-12 and abs(r["translated_acc"] - 1.0) < 1e-12 \
        and abs(r["gap"] - 0.5) < 1e-12, (
        f"the gap is translated minus native over the MATCHED concepts: expected 0.5, got {r['gap']}. "
        "Averaging every translated item against every native item mixes in unmatched concepts."
    )
    both = origin_gap(items, results)
    assert both["n_pairs"] == 3 and both["diffs"].tolist() == [1, 0, -1], (
        f"lang=None selects every language: expected 3 pairs and diffs [1, 0, -1], got "
        f"{both['n_pairs']} and {both['diffs'].tolist()}"
    )
    print("exercise 9 looks right: paired by concept, unmatched items ignored")

In [ ]:
if __name__ == "__main__":
    _try("exercise 9", _check_origin_gap)

Quantify it. For each system, the mean translated-minus-native gap with a bootstrap interval
over concepts, next to the bonus that was planted. Then the artefact's effect on the headline:
the paired `A - B` gap on the translated items against the same gap on the native items.

In [ ]:
def _show_translation_gap() -> None:
    print(f"{'system':7} {'pairs':>6} {'native':>7} {'translated':>11} {'gap':>7}  {'95% interval':>17}  {'planted':>8}  detected")
    diffs = {}
    for s in ("A", "B"):
        r = origin_gap(ITEMS, RESULTS[s])
        diffs[s] = r["diffs"]
        _, lo, hi = bootstrap_ci(r["diffs"], seed=SEED)
        print(f"{s:7} {r['n_pairs']:6d} {r['native_acc']:7.3f} {r['translated_acc']:11.3f} {r['gap']:+7.3f}  "
              f"{f'[{lo:+.3f}, {hi:+.3f}]':>17}  {TRANS_BONUS[s]:+8.3f}  {'yes' if lo > 0 or hi < 0 else 'no'}")
    print("\nthe headline gap between the systems, on each half of the benchmark (paired, A minus B):")
    for label, items in (("translated", TRANSLATED), ("native", NATIVE)):
        d, lo, hi = paired_diff_ci({it.item_id: RESULTS["A"][it.item_id] for it in items},
                                   {it.item_id: RESULTS["B"][it.item_id] for it in items}, seed=SEED)
        print(f"  {label:10} {d:+.3f}  [{lo:+.3f}, {hi:+.3f}]")
    print("\nThe difference between those two lines is the artefact working on the leaderboard.")

In [ ]:
if __name__ == "__main__":
    _try("the artefact, measured", _show_translation_gap,
         needs=("exercise 3", "exercise 5", "exercise 6", "exercise 9"))

## 13. Exercise 10 — how many items before a ranking holds still?

Two languages whose accuracies differ by a few points will swap places on one sample and swap
back on the next, unless each sample is large. Here you measure it. Take a large **pool** of
0/1 outcomes for each of two languages, whose full-pool means define the *true* order. Draw
`n` items at random **without replacement** from each pool, compare the two sample means, and
count how often the sampled order **disagrees** with the true one, over `n_trials` draws. A
tie in the sample means counts as a disagreement, since it does not reproduce the ordering.
Underpowered comparisons are common in NLP (Card et al., in `claims.yaml`).

<details><summary>💡 Hint 1 — what to think about</summary>

What is "the true order" here, and what should happen when the two pools have the same mean?
If `n` equals the size of a pool, how can the sample order differ from the pool order?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Fix the true sign from the full pools and refuse a tie or an `n` larger than a pool. Make one
generator from `seed`. In each trial draw `n` distinct indices from each pool, compare the
two sample means, and count the trial unless the sign strictly matches the true one. Return
the count divided by `n_trials`.
</details>

In [ ]:
def rank_flip_rate(pool_x: np.ndarray, pool_y: np.ndarray, n: int, n_trials: int, seed: int) -> float:
    """Fraction of trials whose sampled ranking of x against y differs from the pools' ranking.

    The true order is the sign of `pool_x.mean() - pool_y.mean()`; raise `ValueError` if it is
    zero or if `n` exceeds the size of either pool. Each trial draws `n` items WITHOUT
    replacement from each pool, and counts as a flip unless the sample means differ in the same
    direction (equal sample means count as a flip). The same `seed` gives the same answer.

    Example:
        >>> x = np.array([1] * 8 + [0] * 2); y = np.array([1] * 2 + [0] * 8)
        >>> rank_flip_rate(x, y, 10, 5, seed=0)     # n is the whole pool: the sample IS the pool
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_rank_flip_rate() -> None:
    with _time_limit(10, "rank_flip_rate"):
        _check_rank_flip_rate_cases()
    print("exercise 10 looks right: no flips when the sample is the pool, none when the gap is huge")


def _check_rank_flip_rate_cases() -> None:
    x = np.array([1] * 8 + [0] * 2)
    y = np.array([1] * 2 + [0] * 8)
    try:
        rank_flip_rate(x, y, 11, 5, seed=0)
    except ValueError:
        pass
    else:
        raise AssertionError("n = 11 cannot be drawn without replacement from a pool of 10: raise "
                             "ValueError before sampling, rather than drawing until 11 distinct items turn up")
    fifty_x, fifty_y = np.array([1] * 30 + [0] * 20), np.array([1] * 20 + [0] * 30)
    assert rank_flip_rate(fifty_x, fifty_y, 50, 20, seed=0) == 0.0, (
        "n = 50 drawn WITHOUT replacement from a pool of 50 is the whole pool every time, so nothing "
        "can flip; a nonzero rate here means items were drawn with replacement"
    )
    assert rank_flip_rate(x, y, 10, 20, seed=0) == 0.0, (
        "when n is the whole pool every sample IS the pool, so the order can never flip: the rate "
        "must be exactly 0.0. Sampling with replacement, or a strict '>' that mishandles the sign, "
        "breaks this."
    )
    big_x = (np.random.default_rng(1).random(2000) < 0.8).astype(int)
    big_y = (np.random.default_rng(2).random(2000) < 0.4).astype(int)
    far = rank_flip_rate(big_x, big_y, 200, 60, seed=3)
    assert far == 0.0, f"0.8 against 0.4 at n=200 never flips in practice, got {far}"
    try:
        rank_flip_rate(x, x.copy(), 5, 10, seed=0)
    except ValueError:
        pass
    else:
        raise AssertionError("two pools with equal means have no true order: raise ValueError")

In [ ]:
if __name__ == "__main__":
    _try("exercise 10", _check_rank_flip_rate)

Now measure the curve for every pair of adjacent languages, ranked by system `A`'s planted
accuracy on translated items (the pools are drawn from those planted probabilities, large enough
to sample from). For each pair: the measured flip rate at each `n`, the smallest `n` from which
it stays at or below five percent (`stable_n`, given), and the classical normal-approximation
sample size for the same gap (`analytic_n`, given) as a cross-check. The analytic figure ignores
that the pool is finite, which matters at the largest `n`.

In [ ]:
POOL_SIZE = 6000
N_GRID = (25, 50, 100, 200, 400, 800, 1600, 3200)
FLIP_TRIALS = 200


def make_pool(p: float, size: int, seed: int) -> np.ndarray:
    """`size` 0/1 outcomes with probability `p` of a 1."""
    return (np.random.default_rng(seed).random(size) < p).astype(np.int64)


def stable_n(curve: dict[int, float], threshold: float) -> int | None:
    """The smallest n at which the flip rate is <= `threshold` and stays so at every larger n."""
    ok = None
    for n in sorted(curve, reverse=True):
        if curve[n] > threshold:
            break
        ok = n
    return ok


def analytic_n(p1: float, p2: float, target: float = 0.05) -> int:
    """Items per language for the normal approximation to put the flip probability at `target`."""
    z = statistics.NormalDist().inv_cdf(1 - target)
    var = p1 * (1 - p1) + p2 * (1 - p2)
    return math.ceil(z * z * var / (p1 - p2) ** 2)


def _show_flip_curves() -> None:
    order = sorted(LANGS, key=lambda lang: -true_p("A", lang, "translated"))
    print(f"{'pair':16} {'gap':>6}  " + " ".join(f"{n:>5}" for n in N_GRID) + f"  {'stable n':>9} {'analytic n':>11}")
    for i, (lo_lang, hi_lang) in enumerate(zip(order, order[1:])):
        px = make_pool(true_p("A", lo_lang, "translated"), POOL_SIZE, SEED + 200 + i)
        py = make_pool(true_p("A", hi_lang, "translated"), POOL_SIZE, SEED + 300 + i)
        curve = {n: rank_flip_rate(px, py, n, FLIP_TRIALS, seed=SEED + n) for n in N_GRID}
        gap = abs(px.mean() - py.mean())
        stable = stable_n(curve, 0.05)
        print(f"{lo_lang + ' / ' + hi_lang:16} {gap:6.3f}  " + " ".join(f"{curve[n]:5.2f}" for n in N_GRID)
              + f"  {(str(stable) if stable else '> ' + str(N_GRID[-1])):>9} {analytic_n(px.mean(), py.mean()):11d}")
    print(f"\nflip rate over {FLIP_TRIALS} draws per cell. The smaller the gap, the more items each language")
    print("needs before its rank stops depending on which items happened to be drawn.")

In [ ]:
if __name__ == "__main__":
    _try("the items-per-language curve", _show_flip_curves, needs=("exercise 10",))

## 14. Ground truth: what the planted numbers say

This is a simulation, so the answer is on the table. For every system, origin and language,
here is the planted accuracy next to your bootstrap interval from the items the benchmark
actually contains. A correct 95% interval should contain the truth about nineteen times in
twenty; the cell prints what that means for this many intervals. A run can land a few short
by chance (the smallest languages have few items, and an interval's ends move in steps of one
item); a count near half would point at a wrong instrument.

In [ ]:
def _show_ground_truth() -> None:
    inside = total = 0
    print(f"{'system':7} {'origin':11} {'language':8} {'items':>6} {'planted':>8} {'measured':>9}  {'95% interval':>16}  contains")
    for s in ("A", "B"):
        for origin, items in (("translated", TRANSLATED), ("native", NATIVE)):
            for lang in LANGS:
                sel = subset(items, lang=lang)
                m, lo, hi = bootstrap_ci(outcomes(RESULTS[s], sel), seed=SEED)
                truth = true_p(s, lang, origin)
                ok = lo <= truth <= hi
                inside += ok
                total += 1
                print(f"{s:7} {origin:11} {lang:8} {len(sel):6d} {truth:8.3f} {m:9.3f}  {f'[{lo:.3f}, {hi:.3f}]':>16}  {'yes' if ok else 'NO'}")
    print(f"\n{inside} of {total} intervals contain the planted accuracy; nineteen in twenty would be "
          f"≈ {0.95 * total:.1f}.")

In [ ]:
if __name__ == "__main__":
    _try("planted against measured", _show_ground_truth, needs=("exercise 3",))

## 15. Common mistakes

- **Fertility per byte.** UTF-8 already charges scripts differently, so a per-byte figure
  divides part of the tax out. Compare per word (when the text is parallel) or per character.
- **Comparing raw token counts of different content.** Fertility is a rate. Tokens for
  different sentences say nothing about the language.
- **Reporting the accuracy without the items behind it.** A language with few items and one with
  many need very different error bars.
- **Resampling the wrong unit.** The random draw is the question, so resample items. Resampling
  the six language accuracies measures the spread of the languages, not the noise in each.
- **Comparing two systems on different items**, or pairing by list position when the order
  differs. Align on item id and refuse a mismatch.
- **Leading with one aggregate.** Macro and micro answer different questions; name the weighting,
  and list every language whose interval reaches chance.
- **Treating a translated benchmark as a native one.** Pair native and translated items of the
  same concept, and quote the gap with its interval.
- **Reading a rank as more stable than its sample size allows.** A ranking of two nearby
  languages needs the item count the curve above says it needs.

In [ ]:
def _show_pair_counts() -> None:
    print("how precisely each language's items pin down an accuracy, at each benchmark size:")
    print(f"{'language':8} {'items':>6} {'formula half-width at 0.5':>26} {'bootstrap half-width, A':>24} "
          f"{'bootstrap half-width, B':>24}")
    measured = {s: language_intervals(s, TRANSLATED) for s in ("A", "B")}
    for lang in LANGS:
        n = len(subset(TRANSLATED, lang=lang))
        half = {s: (measured[s][lang][2] - measured[s][lang][1]) / 2 for s in measured}
        print(f"{lang:8} {n:6d} {1.96 * 0.5 / math.sqrt(n):26.3f} {half['A']:24.3f} {half['B']:24.3f}")
    print("\nThe formula is the normal approximation 1.96 * 0.5 / sqrt(n), largest at accuracy 0.5. An")
    print("accuracy nearer 0 or 1 narrows the interval, and on a few dozen items the bootstrap's ends move")
    print("in steps of 1/n, so it can come out a little wider or narrower than the formula says.")

In [ ]:
if __name__ == "__main__":
    _try("what n buys", _show_pair_counts, needs=("exercise 3",))

## 16. Self-check

1. A colleague divides each script's token count by its UTF-8 byte count and ranks the six
   languages by that figure. What is wrong with the ranking?
   - (a) nothing: the tokenizer works on bytes, so bytes are the natural unit
   - (b) UTF-8 already spends a different number of bytes per character on each script, so the
         figure mixes the encoding's cost with the tokenizer's; on parallel text, tokens per word
         (or per character) measure the tax
   - (c) tokenizers work on characters, so bytes are the wrong thing to count
   - (d) the byte count is the same in every script

2. One language costs twice as many tokens per question as a reference, and the system is right
   on half as many of its questions. Roughly how does its cost per correct answer compare?
   - (a) about the same  - (b) about twice  - (c) about four times  - (d) about a quarter

3. A language contributes 30 items to a benchmark. To say how precisely those 30 items pin down
   the system's accuracy in that language, which is resampled?
   - (a) the accuracies of all the languages, with replacement
   - (b) the tokens in each item
   - (c) the bootstrap means, again
   - (d) the 30 items, with replacement

4. System X wins on the micro-average and system Y on the macro-average. The most defensible
   report is:
   - (a) neither ranking stands alone: give per-language results with intervals, and say
         which weighting each ranking assumes
   - (b) X, because micro uses more items
   - (c) Y, because macro is always fairer
   - (d) whichever aggregate the system's authors prefer

5. A system scores higher on the translated version of a benchmark than on the native items for
   the same concepts. Which analysis can separate a real artefact from sampling noise?
   - (a) compare the two headline averages by eye
   - (b) take the larger benchmark's word for it
   - (c) the per-concept translated-minus-native difference, with a bootstrap interval over concepts
   - (d) retrain the tokenizer

6. Two languages swap places between random samples of 100 items but not between samples of
   5000. What does that tell you?
   - (a) the languages are equally good
   - (b) the true gap is small relative to the noise at 100 items, so the ranking there is
         decided by which items were drawn; the sample size needed grows as the gap shrinks
   - (c) the system is unreliable
   - (d) the benchmark contains an error

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands now:
# each one already printed its feedback in its own cell above, so the board is current even if
# you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check, the exercises that check relies on)
        ("exercise 1", _check_fertility, ()),
        ("exercise 2", _check_cost_and_window, ()),
        ("exercise 3", _check_bootstrap_ci, ()),
        ("exercise 4", _check_macro_ci, ()),
        ("exercise 5", _check_align_paired, ()),
        ("exercise 6", _check_paired_diff_ci, ("exercise 5",)),
        ("exercise 7", _check_aggregate, ()),
        ("exercise 8", _check_weak_languages, ()),
        ("exercise 9", _check_origin_gap, ()),
        ("exercise 10", _check_rank_flip_rate, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    print(f"lesson wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s\n")
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong is:
    # in a script or under CI it ends the run non-zero. Inside a notebook kernel the board above
    # has already said so, in a line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

## What you built, and where it goes next

Nine instruments for evaluating across languages, each checked against a system whose true
accuracy you could look up: fertility, cost per correct answer, an item-level interval, a
macro-average interval, a paired comparison that refuses mismatched items, macro and micro
aggregation, a chance-level flag, a translation-artefact gap and an items-per-language curve.
The same instruments apply unchanged to a real model's answers on a real benchmark: swap the
planted results for what the model returned, keep the item ids, and everything above still
applies. The papers behind each effect are in `claims.yaml`, with the date each was read.

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.